<a href="https://colab.research.google.com/github/jasman5/UCS547-Accelerated-Data-Science/blob/main/assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assignment II – CUDA C++ Programming

1. Identify !, %, and %% used in cell in Google Colab.

In [6]:
#  ! → Shell / System Command (os commands)
#Runs Linux terminal commands inside a notebook.
!ls
!pip install numpy

sample_data


In [4]:
!nvidia-smi

Fri Jan 30 10:43:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# % → Line Magic Command. one line control
# Applies to one line only . Used for timing, environment, plotting, etc.
%time x = [i*i for i in range(10000)]
%pwd
%matplotlib inline

CPU times: user 468 µs, sys: 0 ns, total: 468 µs
Wall time: 472 µs


In [8]:
# %% → Cell Magic Command. full cell control
# Applies to entire cell . Used for timing, bash scripts, file writing, etc.
%%bash
echo "Hello from bash"

Hello from bash


In [10]:
%%time
for i in range(1000000):
    pass

CPU times: user 37.9 ms, sys: 1.02 ms, total: 39 ms
Wall time: 39.5 ms


2. Identify all key nvidia-smi commands with multiple options

In [12]:
#nvidia-smi = NVIDIA System Management Interface (GPU monitoring tool)
!nvidia-smi

Fri Jan 30 10:48:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
#Live monitoring
!nvidia-smi -l 1

Fri Jan 30 10:49:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
#GPU utilization
!nvidia-smi --query-gpu=utilization.gpu --format=csv

utilization.gpu [%]
0 %


In [15]:
#Memory usage
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

memory.used [MiB], memory.total [MiB]
0 MiB, 15360 MiB


In [16]:
#Temperature & power
!nvidia-smi --query-gpu=temperature.gpu,power.draw --format=csv

temperature.gpu, power.draw [W]
42, 9.83 W


In [17]:
#Running GPU processes
!nvidia-smi pmon

# gpu         pid   type     sm    mem    enc    dec    jpg    ofa    command 
# Idx           #    C/G      %      %      %      %      %      %    name 
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0          -     -      -      -      -      -      -      -    -              
    0

In [ ]:
#Select specific GPU
!nvidia-smi -i 0
#Help
!nvidia-smi --help
#Log output
!nvidia-smi -l 1 > gpu_log.txt

3. Debug common CUDA errors

In [20]:
#Zero Output Causes
#Kernel not launched
#No cudaDeviceSynchronize()
#Threads > data size
!cudaDeviceSynchronize();

/bin/bash: -c: line 1: syntax error near unexpected token `;'
/bin/bash: -c: line 1: `cudaDeviceSynchronize();'


In [21]:
#Incorrect Indexing
# due to Wrong global index calculation
int idx = blockIdx.x * blockDim.x + threadIdx.x;

SyntaxError: invalid syntax (ipython-input-608295273.py, line 3)

In [ ]:
#PTX Errors
#GPU architecture mismatch
#Wrong CUDA version
#Old compiler
!nvcc -arch=sm_75 file.cu

4. Write a CUDA C/C++ program to demonstrate GPU kernel execu on and thread indexing.
a. Launch a CUDA kernel using: 1 block and 8 threads
b. Each thread must print: Hello from GPU thread <global_thread_id>
c. Compute the global thread ID using: global_thread_id = blockIdx.x * blockDim.x +
threadIdx.x
d. Clearly separate: Host code (CPU) & Device code (GPU kernel)

In [43]:
%%writefile hello_safe.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void kernel(int *out) {
    int id = threadIdx.x;
    out[id] = id;
}

int main() {
    int h[8];
    int *d;

    cudaMalloc(&d, 8 * sizeof(int));
    kernel<<<1, 8>>>(d);
    cudaMemcpy(h, d, 8 * sizeof(int), cudaMemcpyDeviceToHost);

    for (int i = 0; i < 8; i++)
        printf("Hello from GPU thread %d\n", h[i]);

    cudaFree(d);
    return 0;
}


Writing hello_safe.cu


In [44]:
!nvcc hello_safe.cu -o hello_safe
!./hello_safe

Hello from GPU thread 0
Hello from GPU thread 0
Hello from GPU thread 0
Hello from GPU thread 0
Hello from GPU thread 0
Hello from GPU thread 0
Hello from GPU thread 0
Hello from GPU thread 0


5. Write a CUDA program to demonstrate host and device memory separa on.
a. Create an integer array of size 5 on the host (CPU).
b. Allocate corresponding memory on the device (GPU) using cudaMalloc().
c. Copy data from host to device using cudaMemcpy().
d. Launch a kernel where GPU threads print values from device memory.
e. Copy the data back from device to host and print it on CPU.

In [45]:
%%writefile memory_separation.cu
#include <stdio.h>
#include <cuda_runtime.h>

// ---------------- DEVICE CODE (GPU) ----------------
__global__ void readDeviceArray(int *d_arr) {
    int idx = threadIdx.x;
    d_arr[idx] = d_arr[idx] + 1;   // simple operation
}

// ---------------- HOST CODE (CPU) ----------------
int main() {

    // a) Host array
    int h_arr[5] = {10, 20, 30, 40, 50};
    int *d_arr;

    // b) Allocate device memory
    cudaMalloc((void**)&d_arr, 5 * sizeof(int));

    // c) Copy Host → Device
    cudaMemcpy(d_arr, h_arr, 5 * sizeof(int), cudaMemcpyHostToDevice);

    // d) Launch kernel (1 block, 5 threads)
    readDeviceArray<<<1, 5>>>(d_arr);
    cudaDeviceSynchronize();

    // e) Copy Device → Host
    cudaMemcpy(h_arr, d_arr, 5 * sizeof(int), cudaMemcpyDeviceToHost);

    // Print result on CPU
    printf("CPU Output:\n");
    for (int i = 0; i < 5; i++) {
        printf("%d ", h_arr[i]);
    }

    cudaFree(d_arr);
    return 0;
}


Writing memory_separation.cu


In [46]:
!nvcc memory_separation.cu -o memory_separation
!./memory_separation


CPU Output:
10 20 30 40 50 

6. Compare CPU times of List/tuple with Numpy arrays.

In [30]:
import time
import numpy as np

n = 1000000

# List
lst = list(range(n))
start = time.time()
[x * 2 for x in lst]
print("List time:", time.time() - start)

# Tuple
tpl = tuple(range(n))
start = time.time()
[x * 2 for x in tpl]
print("Tuple time:", time.time() - start)

# NumPy
arr = np.array(range(n))
start = time.time()
arr * 2
print("NumPy time:", time.time() - start)


List time: 0.10503721237182617
Tuple time: 0.061689138412475586
NumPy time: 0.0013737678527832031
